# HumanoidBench — 多任务 × 多策略 MuJoCo 一键预览

_Multi-task × multi-policy one-click MuJoCo preview_

基于 [carlosferrazza/humanoid-bench](https://github.com/carlosferrazza/humanoid-bench) (RSS 2024)，在 **MuJoCo 3.1.6** 中预览 **Unitree H1 / H1Hand / G1** 在 27 个 RL 任务上的场景与策略。

_Built on humanoid-bench (RSS 2024). Previews 27 RL tasks for Unitree H1 / H1Hand / G1 inside the native MuJoCo 3.1.6 viewer._

## 两轴一键预览 / Two-axis one-click preview

| 轴 / Axis | 内容 / Coverage | 章节 / Sections |
|---|---|---|
| **任务轴 / Task** | Locomotion (12) · Manipulation (15) · 机器人变体 / Robot variants | §3, §4, §5 |
| **策略轴 / Policy** | `random` / `zero` / 官方 reach 低层技能 / 社区 DR.Q ckpt | §6, §7, §8, §9 |

## 流程概览 / Pipeline

```
Unitree H1 MJCF → MuJoCo 仿真 → gym 接口 → RL 训练 (Dreamer/TDMPC2/SAC/PPO) → policy → 本 Notebook 预览
                                                                                  ↑
                                                       默认走 random/zero/reach-skill/DR.Q，跳过训练
                                                       Default path: random/zero/reach-skill/DR.Q — no training
---

## 1. 初始化子模块

_Init submodule_

In [ ]:
!git submodule update --init --recursive dependencies/humanoid-bench

## 2. 创建并安装 conda 环境（幂等）

_Create and install the conda env (idempotent)_

官方要求 **Python 3.11**，`setup.py` 钉死 `mujoco==3.1.6` / `dm_control==1.0.20` / `torch==2.3.1`。
_Upstream requires Python 3.11 and pins `mujoco==3.1.6` / `dm_control==1.0.20` / `torch==2.3.1` in `setup.py`._

> ⚠️ **独立 env 是刚需 / Isolated env is mandatory**
>
> humanoid-bench 的 mujoco/torch 版本会和 KungfuBot 的 `kungfubot` env（`mujoco==3.2.3` / `numpy==1.24.4`）冲突，互装会静默降级、两个 notebook 都打坏。
>
> _humanoid-bench's mujoco/torch versions clash with KungfuBot's `kungfubot` env (`mujoco==3.2.3` / `numpy==1.24.4`) — installing into the same env will silently downgrade and break both notebooks._

> 💡 **预览不装训练栈 / Preview ≠ training stack**
>
> 跑预览**不需要**装 jax / dreamer / jaxrl / tdmpc / SB3-PPO；要训练时按 §9 / 上游 README 增量补装。
>
> _Preview alone does NOT require jax / dreamer / jaxrl / tdmpc / SB3-PPO — install training deps on demand later._

In [ ]:
%%bash
set -e
source "$(conda info --base)/etc/profile.d/conda.sh"

ENV=humanoidbench
if conda env list | awk '{print $1}' | grep -qx "$ENV"; then
    echo "✅ conda env '$ENV' already exists, skipping create."
else
    echo "📦 creating conda env '$ENV' (python=3.11) ..."
    conda create -n "$ENV" python=3.11 -y
fi

echo "📦 installing humanoid_bench (editable) ..."
conda run -n "$ENV" --no-capture-output pip install -e ./dependencies/humanoid-bench

conda run -n "$ENV" --no-capture-output pip install -q ipykernel
conda run -n "$ENV" python -m ipykernel install --user --name="$ENV" --display-name "Python ($ENV)" >/dev/null 2>&1 || true

echo "🎉 done. 后面的格子用 conda run -n $ENV 直接跑。"

**关键依赖 / Key dependencies**：`gymnasium==0.29.1` · `mujoco==3.1.6` · `mujoco-mjx==3.1.6` · `dm_control==1.0.20` · `torch==2.3.1` · `brax==0.9.4` · `gymnax==0.0.8` · `opencv-python`

---

## 一键预览脚本说明 / Native viewer script

`scripts/native_viewer.py` 直接调 `mujoco.viewer.launch_passive`，得到 **高清 + 鼠标交互 + 阴影抗锯齿** 的 MuJoCo 原生窗口，比官方 `test_env.py` 的 cv2 240×240 小窗清晰得多。

_`scripts/native_viewer.py` calls `mujoco.viewer.launch_passive` directly — high-resolution, interactive, AA shadows. Far better than upstream `test_env.py`'s 240×240 cv2 window._

### MuJoCo 窗口按键 / Keyboard cheat-sheet

| 按键 / Key | 功能 / Action |
|---|---|
| `Space` | 暂停 / 继续 · pause / resume |
| 鼠标左键拖拽 / LMB drag | 旋转视角 · orbit camera |
| 鼠标右键拖拽 / RMB drag | 平移 · pan |
| 滚轮 / scroll | 缩放 · zoom |
| 双击物体 / double-click | 聚焦相机 · focus camera |
| `Ctrl + drag` | 对机器人施加外力 · apply external force |
| `[` / `]` | 切换 named camera · cycle named cameras |
| `Esc` / 关窗 / close | 退出 · quit |

### 脚本参数 / Script arguments

```
python scripts/native_viewer.py --env <env-id> --action {random,zero} [--fps 30]
```

- `--action random`（默认 / default）：机器人在随机动作下抽搐摔倒，仅为看场景。
  _Random torque each step — robot twitches and falls. Use it to inspect the scene only._
- `--action zero`：所有关节力矩为 0，看初始 keyframe + 自由落体。
  _Zero torque — shows the initial keyframe and lets gravity reveal the rest pose._

---

# 第一轴：任务一键预览 / Axis 1 · one-click per task

## 3. Locomotion 套餐（12 任务）

_Locomotion bundle (12 tasks) — no manipulation goal, whole-body control only. obs≈151, act≈61._

无操作目标，纯全身运动控制。obs 维度 ~151，act 维度 ~61。

In [ ]:
# 🚶 走 / Walk
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-walk-v0

In [ ]:
# 🏃 跑 / Run
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-run-v0

In [ ]:
# 🧍 站立 / Stand
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-stand-v0 --action zero

In [ ]:
# 🪑 坐下 / Sit (simple / hard)
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-sit_simple-v0

In [ ]:
# ⚖️ 平衡 / Balance (hard)
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-balance_hard-v0

In [ ]:
# 🐛 爬行 / Crawl
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-crawl-v0

In [ ]:
# 🏗️ 跨栏 / Hurdle
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-hurdle-v0

In [ ]:
# 🪜 阶梯 / Stair
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-stair-v0

In [ ]:
# 🛝 滑梯 / Slide
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-slide-v0

In [ ]:
# 🪵 杆上平衡 / Pole
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-pole-v0

In [ ]:
# 🧭 迷宫 / Maze
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-maze-v0

In [ ]:
# ✋ 触达 / Reach（双手伸向目标 · two hands reaching target）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-reach-v0

---

## 4. Manipulation 套餐（15 任务）

_Manipulation bundle (15 tasks) — whole-body + Shadow Hand. Kitchen / Cube / Bookshelf use **fixed initial state** (reproducible eval)._

全身 + Shadow Hand 操作。Kitchen / Cube / Bookshelf 是 **固定初始状态**（reproducible 评测友好）。

In [ ]:
# 📦 推箱子 / Push
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-push-v0

In [ ]:
# 🧊 魔方 / Cube（shadow hand 精密操作 · precision in-hand manipulation）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-cube-v0

In [ ]:
# 🚪 开门 / Door
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-door-v0

In [ ]:
# 📨 取放包裹 / Package
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-package-v0

In [ ]:
# 🚛 卡车装载 / Truck
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-truck-v0

In [ ]:
# 🏀 篮球 / Basketball
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-basketball-v0

In [ ]:
# 🍳 厨房 / Kitchen（橱柜+烤箱+水槽 · cabinet+oven+sink，固定初始 · fixed init）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-kitchen-v0

In [ ]:
# 🗄️ 橱柜 / Cabinet
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-cabinet-v0

In [ ]:
# 📚 书架（简单） / Bookshelf simple（固定初始 · fixed init）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-bookshelf_simple-v0

In [ ]:
# 📚 书架（困难） / Bookshelf hard（固定初始 · fixed init）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-bookshelf_hard-v0

In [ ]:
# 🪟 擦窗 / Window
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-window-v0

In [ ]:
# 🥄 勺子 / Spoon
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-spoon-v0

In [ ]:
# 🔩 插入（标准孔） / Insert normal
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-insert_normal-v0

In [ ]:
# 🔩 插入（小孔） / Insert small（更难 · tighter tolerance）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-insert_small-v0

In [ ]:
# 🏋️ 高栏（强化手） / Highbar hard（需 h1strong · stronger grip required）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1strong-highbar_hard-v0

---

## 5. 机器人变体对比

_Robot variant comparison — same task, different robot prefix._

同一任务（如 `walk`）换前缀即可切换机器人，对比「带手 vs 不带手」「H1 vs G1」的视觉/动作空间区别。
_Swap the prefix to compare hand vs no-hand, or H1 vs G1, on the same task._

| 前缀 / Prefix | 机器人 / Robot | act 维度 / dim | 适用任务 / Coverage |
|---|---|---|---|
| `h1-` | H1 无手 / no hands | ~19 | 纯 locomotion 12 任务 / locomotion-only |
| `h1hand-` | H1 + 双 Shadow Hand | ~61 | 全 27 任务 / all tasks |
| `h1touch-` | H1 + 触觉手 / tactile hands | ~61 | 含 tactile（需 `test_env.py --obs_wrapper True --sensors proprio,image,tactile`） |
| `h1strong-` | H1 + 加强手 / stronger grip | ~61 | `highbar_hard` 专用 / dedicated |
| `h1simplehand-` | H1 + 低维手 / low-dim hands | ~25 | `pole` 等 |
| `g1-` | Unitree G1 三指手 / 3-finger hands | ~37 | 6 个 locomotion / locomotion only |

### h1 (无手) vs h1hand (带 Shadow Hand) 走路对比

_h1 (no hands) vs h1hand (Shadow Hand) walking comparison._

In [ ]:
# h1-walk-v0：纯腿 + 上身，无手部 DOF · legs + torso only, no hand DoF
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1-walk-v0

In [ ]:
# h1hand-walk-v0：带双 Shadow Hand · with two Shadow Hands, independent finger DoF
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-walk-v0

### Unitree G1（三指手）走路

_Unitree G1 (3-finger hands) walking._

In [ ]:
# g1-walk-v0
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env g1-walk-v0

In [ ]:
# g1-push-v0：G1 推箱子 · G1 push
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env g1-push-v0

---

# 第二轴：策略一键预览 / Axis 2 · one-click per policy

## 6. 策略 A — Random action（默认）

_Policy A — Random action (the default driver used in §3 / §4 / §5)._

`env.action_space.sample()` 每步随机采样动作。机器人原地抽搐摔倒 —— **只看场景**。
_`env.action_space.sample()` per step. Robot twitches and falls — **scene preview only**._

## 7. 策略 B — Zero action（看初始姿态 / 自由落体）

_Policy B — Zero action (initial keyframe + free-fall under gravity)._

所有关节扭矩 = 0。看 keyframe 给的初始姿态，机器人会因重力慢慢倒下 —— **看机器人静态**。
_All joint torques = 0. Reveals the keyframe rest pose, then gravity takes over — **static-robot inspection**._

In [ ]:
# Zero action：看 h1hand-stand-v0 的初始 keyframe · initial keyframe of stand
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py --env h1hand-stand-v0 --action zero

## 8. 策略 C — 官方自带 reach 低层技能

_Policy C — Upstream built-in low-level reach skill (~1M-param torch MLP)._

`dependencies/humanoid-bench/data/reach_two_hands/` 自带 PyTorch **小 MLP**（~1M 参数），用作 hierarchical primitive。
_Located in `dependencies/humanoid-bench/data/reach_two_hands/`, a small PyTorch MLP used as a hierarchical primitive._

启动方式：`scripts/native_viewer.py --policy_type reach_double_relative --policy_path ...` —— env 内部把 act 维度从 61 降到 6（双手 3D target），reach skill 翻译到 61 维关节扭矩。
_Launch with `scripts/native_viewer.py --policy_type reach_double_relative --policy_path ...` — the env internally shrinks the action space from 61 to 6 (two 3-D hand targets); the reach skill expands it back to 61 joint torques._

- **能看到 / What you'll see**：手臂有意义地伸向目标点 · arms reaching the target meaningfully
- **看不到 / What you won't see**：躯干/腿协调（仍随机） · torso/legs are still random
- **act 维度变化 / Action-space shrink**：61 → 6 (hierarchical wrapper)

> ⚠️ **不是所有 task 都支持 reach wrapper / Not every task supports the reach wrapper**
>
> Wrapper 要求 task class 定义 `htarget_low` / `htarget_high`（手部目标的采样范围）。
> _The wrapper expects the task class to declare `htarget_low` / `htarget_high` (sampling range for hand targets)._
>
> | | Tasks |
> |---|---|
> | ✅ **支持 / Supported** | `reach` · `walk` · `run` · `stand` · `sit` · `balance` · `stair` · `slide` · `hurdle` · `crawl` · `maze` · `pole` · `package` · `truck` · `bookshelf` · `push` |
> | ❌ **不支持 / Not supported** | `cabinet` · `door` · `cube` · `kitchen` · `basketball` · `window` · `spoon` · `insert` · `highbar` |
>
> 不支持的 task 跑这个 wrapper 会报 `AttributeError: ... has no attribute 'htarget_low'`。
> _Unsupported tasks raise `AttributeError: ... has no attribute 'htarget_low'`. `native_viewer.py` catches this and prints the supported list._

> ✅ **现已支持原生 MuJoCo viewer / Native MuJoCo viewer supported**
>
> 之前必须用官方 `test_env.py`（cv2 240×240 小窗）；`native_viewer.py` 扩了 `--policy_path/--mean_path/--var_path/--policy_type` 后两者皆可。
>
> _Previously required upstream `test_env.py` (240×240 cv2). `native_viewer.py` now accepts `--policy_path/--mean_path/--var_path/--policy_type` and renders the same hierarchical env in the high-resolution native viewer._

In [ ]:
# 🦾 h1hand-push + 双手 reach 低层技能 (native viewer)
#     two-hand reach skill on push — high-res MuJoCo viewer
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py \
        --env h1hand-push-v0 \
        --policy_path dependencies/humanoid-bench/data/reach_two_hands/torch_model.pt \
        --mean_path   dependencies/humanoid-bench/data/reach_two_hands/mean.npy \
        --var_path    dependencies/humanoid-bench/data/reach_two_hands/var.npy \
        --policy_type reach_double_relative

In [ ]:
# 📨 h1hand-package + 双手 reach (native viewer)
#     two-hand reach skill on package — 看双手伸向包裹 · arms aiming at the package
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/native_viewer.py \
        --env h1hand-package-v0 \
        --policy_path dependencies/humanoid-bench/data/reach_two_hands/torch_model.pt \
        --mean_path   dependencies/humanoid-bench/data/reach_two_hands/mean.npy \
        --var_path    dependencies/humanoid-bench/data/reach_two_hands/var.npy \
        --policy_type reach_double_relative

# ⚠️ 注意：不是所有 task 都支持 reach hierarchical wrapper
#     Note: not every task supports the reach hierarchical wrapper
#     ✅ 支持 / Supported: reach · walk · run · stand · sit · balance · stair · slide
#                        hurdle · crawl · maze · pole · package · truck · bookshelf · push
#     ❌ 不支持 / Not supported: cabinet · door · cube · kitchen · basketball · window
#                              spoon · insert · highbar （没定义 htarget_low/high）

## 9. 策略 D — 社区 DR.Q ckpt（28 任务 × 10 seeds 全身策略）

_Policy D — Community DR.Q checkpoints (28 locomotion tasks × 10 seeds, whole-body policies)._

**ICML 2026 论文 DR.Q** 在 HuggingFace 公开了 HumanoidBench locomotion 套件的训练好 ckpt。本项目把 DR.Q 作为 submodule 收编到 `dependencies/dr-q/`，配套 `scripts/drq_viewer.py` 用 minimal eval loader 把它接到 MuJoCo native viewer 上。
_The DR.Q paper (ICML 2026) released trained checkpoints for HumanoidBench's locomotion suite on HuggingFace. This project vendors DR.Q as a submodule under `dependencies/dr-q/` and ships `scripts/drq_viewer.py`, a minimal eval-only loader that wires DR.Q's policy into the native MuJoCo viewer._

| 项 / Item | 值 / Value |
|---|---|
| 模型卡 / Model card | <https://huggingface.co/dmux/DR.Q> |
| 代码 / Code (submodule) | `dependencies/dr-q/` ← <https://github.com/dmksjfl/DR.Q> |
| 算法 / Algorithm | DR.Q (TD3 系 + model-based representation · TD3-family + model-based representation) |
| ✅ 覆盖 / Covered | 28 个 locomotion 任务（h1 / h1hand 各 14 个），每个 10 seeds · 28 locomotion tasks (14 × `h1`, 14 × `h1hand`), 10 seeds each |
| ❌ 不覆盖 / Not covered | manipulation（push / cube / kitchen / cabinet / bookshelf_hard / package / truck / spoon / insert…）|
| 关键文件 / Critical files | `policy.pt` (2.2 MB) · `encoder.pt` (11.1 MB) · `agent_var.npy` (~few KB) |
| 跳过文件 / Skipped files | `buffer_data.npz` (~367 MB) · 各类 `*_target.pt` / `*_optimizer.pt`（仅训练续训用） |
| 报告分数 / Reported score | `h1hand-walk-v0` normalized **512 [371, 652]** @ 1M env steps |
| 本机实测 / Our measurement | ep 1 = **557**（落在 [371, 652] 内）✅ |

### 9.1 拉取 DR.Q submodule（一次性 · one-off）

_Pull the DR.Q submodule (one-off setup)._

`dependencies/dr-q/` 已在本项目 `.gitmodules` 注册，直接初始化即可。
_`dependencies/dr-q/` is already registered in this project's `.gitmodules`, just init it._

In [ ]:
!git submodule update --init --recursive dependencies/dr-q

### 9.2 下载 checkpoint 到 HF 默认 cache

_Download the checkpoint to the HF default cache._

不指定 `--local-dir`，文件落到 `~/.cache/huggingface/hub/models--dmux--DR.Q/snapshots/<sha>/`，`drq_viewer.py` 自动从那里寻址。
_With no `--local-dir`, files land in `~/.cache/huggingface/hub/models--dmux--DR.Q/snapshots/<sha>/`. `drq_viewer.py` resolves the path from there automatically._

只下三件套（**~13 MB**），跳过 `buffer_data.npz`（367 MB）和 `*_target/_optimizer.pt`。
_Pull only the three critical files (**~13 MB**); skip the 367 MB replay buffer and the target/optimizer states (training-only)._

In [ ]:
%%bash
# 新版 HF CLI 入口是 `hf`（老的 `huggingface-cli` 已 deprecated）
# Note: the entry point is now `hf` — `huggingface-cli` is deprecated
conda run -n humanoidbench --no-capture-output pip install -q -U huggingface_hub
conda run -n humanoidbench --no-capture-output \
    hf download dmux/DR.Q \
        --include 'DRQ+HBench-h1hand-walk-v0+0/policy.pt' \
        --include 'DRQ+HBench-h1hand-walk-v0+0/encoder.pt' \
        --include 'DRQ+HBench-h1hand-walk-v0+0/agent_var.npy'

echo "✅ 落地路径 / Landed at:"
find ~/.cache/huggingface/hub/models--dmux--DR.Q -name 'policy.pt' -o -name 'encoder.pt' -o -name 'agent_var.npy' 2>/dev/null

### 9.3 在 MuJoCo native viewer 跑 DR.Q 策略

_Run the DR.Q policy in the native MuJoCo viewer._

`scripts/drq_viewer.py` 干这几件事 / does the following:

1. 在 HF 默认 cache 里定位 `DRQ+HBench-<task>+<seed>/`
   _Locates `DRQ+HBench-<task>+<seed>/` inside the HF default cache._
2. 从 `agent_var.npy` 读 hyperparameters（自动处理老 module 名 `REP` → `DRQ` 的 pickle alias）
   _Reads hyperparameters from `agent_var.npy` (auto-aliases the legacy `REP` → `DRQ` module name baked into the pickle)._
3. 直接构造 `Encoder` + `Policy`（绕开 DR.Q 的 `Agent.load`，这样不需要下 367 MB 的 buffer）
   _Builds `Encoder` + `Policy` directly, bypassing DR.Q's full `Agent.load` so the 367 MB replay buffer is not needed._
4. 用 `policy.act(encoder.zs(state))` 推理，喂给 humanoid-bench env，画到 `mujoco.viewer`
   _Runs `policy.act(encoder.zs(state))` inference, steps the humanoid-bench env, renders to `mujoco.viewer`._

每完成一个 episode 打印 `return`，对比 DR.Q 论文的 normalized score 区间。
_Prints `return` after each episode; cross-check with DR.Q's normalized-score interval._

In [ ]:
# 🚶 h1hand-walk seed 0 — DR.Q trained policy（看真正会走路 · actually walking）
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/drq_viewer.py --task h1hand-walk-v0 --seed 0

### 9.4 换任务 / 换 seed 一键复现

_One-click swap to other tasks or seeds._

`drq_viewer.py` 现在会**自动按需下载** ckpt（每次 ~13 MB 三件套到 HF 默认 cache），换 `--task` / `--seed` 直接跑即可。
_`drq_viewer.py` now **auto-downloads** the ckpt on demand (~13 MB triplet into the HF default cache). Just change `--task` / `--seed` and run._

全 28 个 locomotion 任务 × 10 seeds 都在 [`dmux/DR.Q`](https://huggingface.co/dmux/DR.Q) 上。
_All 28 locomotion tasks × 10 seeds are available at [`dmux/DR.Q`](https://huggingface.co/dmux/DR.Q)._

> 💡 想跳过自动下载（比如断网环境）：加 `--no_download`，缺 ckpt 会直接报错。
> _Add `--no_download` (e.g. offline) — script fails fast instead of fetching._

In [ ]:
# 🏃 h1hand-run seed 0
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/drq_viewer.py --task h1hand-run-v0 --seed 0

In [ ]:
# 🪜 h1-stair seed 0 (无手 H1 上楼梯 · no-hand H1 climbing stairs)
!DISPLAY=:0 conda run -n humanoidbench --no-capture-output \
    python scripts/drq_viewer.py --task h1-stair-v0 --seed 0